# Text-to-image generation

This notebook shows the SDK path for text-to-image generation. It uses **MAI-Image-2e** by default and includes an optional **gpt-image-2** cell.

Before running paid cells, authenticate to Azure (`az login` or managed identity), configure the environment variables described in `notebooks/README.md`, and set `RUN_IMAGE_GENERATION=1`.

In [ ]:
from pathlib import Path
import os
import sys

from IPython.display import Image, display
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "packages" / "t2i_core").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "packages" / "t2i_core" / "src"))
load_dotenv(PROJECT_ROOT / ".env")

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "text_to_image"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_IMAGE_GENERATION = os.getenv("RUN_IMAGE_GENERATION") == "1"
print(f"Output directory: {OUTPUT_DIR}")
print("Paid image calls enabled:", RUN_IMAGE_GENERATION)

In [ ]:
from t2i_core import GPTImageProvider, MAIImageProvider, Settings

settings = Settings()
print("Default MAI efficient deployment:", settings.mai_image_efficient_deployment)
print("Optional GPT image deployment:", settings.gpt_image_deployment)

In [ ]:
prompt = (
    "Photorealistic product hero image of a matte cobalt insulated travel mug "
    "on a warm stone kitchen counter, morning side light, shallow depth of field, "
    "premium ecommerce composition, no text."
)
size = "1024x1024"
quality = "high"

if RUN_IMAGE_GENERATION:
    provider = MAIImageProvider(settings, deployment=settings.mai_image_efficient_deployment)
    try:
        results = await provider.generate(prompt, size=size, quality=quality, n=1)
    finally:
        await provider.aclose()

    output_path = OUTPUT_DIR / "mai-image-2e-hero.png"
    output_path.write_bytes(results[0].image)
    print("Model:", results[0].model)
    print("Estimated SDK cost metadata:", results[0].usage.model_dump())
    display(Image(filename=str(output_path)))
else:
    print("Set RUN_IMAGE_GENERATION=1 to call MAI-Image-2e and save the result.")

## Optional: generate with gpt-image-2

`gpt-image-2` uses the Azure OpenAI Images API. Keep `n` small while developing because each call can incur image-generation costs.

In [ ]:
if RUN_IMAGE_GENERATION:
    provider = GPTImageProvider(settings)
    try:
        results = await provider.generate(prompt, size=size, quality=quality, n=1, background="auto")
    finally:
        await provider.client.close()

    output_path = OUTPUT_DIR / "gpt-image-2-hero.png"
    output_path.write_bytes(results[0].image)
    print("Model:", results[0].model)
    print("Revised prompt:", results[0].revised_prompt)
    print("Estimated SDK cost metadata:", results[0].usage.model_dump())
    display(Image(filename=str(output_path)))
else:
    print("Set RUN_IMAGE_GENERATION=1 to call gpt-image-2.")